# 🫁 Chest X-Ray Attention Training

Training ResNet50/DenseNet121 with Attention (SE, CBAM, ECA) for chest X-ray classification.

**Dataset:** Chest X-Ray (Pneumonia, Covid-19, Tuberculosis, Normal)

**GPU:** Tesla P100 (16GB)


In [ ]:
# Install dependencies
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118 -q
!pip install transformers scikit-learn matplotlib seaborn tqdm opencv-python-headless -q
print('✅ Dependencies installed')

In [ ]:
# Clone repository
!git clone https://github.com/kinhluan/chest-xray-disease-classifier.git
%cd chest-xray-disease-classifier

# Install as package
!pip install -e . -q
print('✅ Repository cloned')

In [ ]:
# Verify GPU
import torch
print(f'PyTorch version: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

In [ ]:
# Download dataset
!kaggle datasets download -d jtiptj/chest-xray-pneumoniacovid19tuberculosis -p data/raw --unzip

# Verify structure
from pathlib import Path
print('\nDataset structure:')
for cls in sorted(Path('data/raw').iterdir()):
    if cls.is_dir():
        count = len(list(cls.glob('*.*')))
        print(f'  {cls.name}/: {count} images')

In [ ]:
# Train Model: ResNet50 + CBAM
!python train.py \
  --data_dir data/raw \
  --model_type resnet \
  --model_name resnet50 \
  --attention cbam \
  --epochs 30 \
  --batch_size 32 \
  --lr 1e-4 \
  --output_dir results/models/resnet50_cbam

print('\n✅ Training complete!')

In [ ]:
# View results
import json
from pathlib import Path

metrics_file = Path('results/models/resnet50_cbam/metrics.json')
if metrics_file.exists():
    with open(metrics_file) as f:
        metrics = json.load(f)
    print('Results:')
    print(f"  Accuracy: {metrics.get('accuracy', 0):.4f}")
    print(f"  F1 Macro: {metrics.get('f1_macro', 0):.4f}")
else:
    print('Metrics file not found')

In [ ]:
# Download results
import zipfile

# Zip results
with zipfile.ZipFile('results.zip', 'w', zipfile.ZIP_DEFLATED) as zipf:
    for file in Path('results').rglob('*'):
        if file.is_file() and file.suffix in ['.pth', '.json', '.txt', '.png']:
            zipf.write(file)

print(f'✅ Results zipped: results.zip')
print(f'   Size: {Path("results.zip").stat().st_size / 1e6:.1f} MB')

# Download from Kaggle
from kaggle_secrets import UserSecretsClient
# Results will be available in Kaggle notebook output